# 🧹 Day 05a: Pandas Data Cleaning — From Messy to Ready

---

## 🎯 What You'll Master Today
- Missing values: fillna, dropna, interpolate
- Duplicates: detection and removal
- Type conversion and string cleaning
- Outlier handling
- pipe() for clean pipelines
- End-to-end cleaning workflow

**Goal:** Clean any messy dataset from scratch in an interview.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  DATA CLEANING CHECKLIST                                         ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. INSPECT:     shape, dtypes, info(), describe(), head()      ║
║  2. MISSING:     isnull().sum(), fillna(), dropna()             ║
║  3. DUPLICATES:  duplicated(), drop_duplicates()                ║
║  4. TYPES:       astype(), to_datetime(), to_numeric()          ║
║  5. STRINGS:     strip(), lower(), replace(), extract()         ║
║  6. OUTLIERS:    IQR method, z-score, clip()                    ║
║  7. VALIDATE:    assert no nulls, correct dtypes, shape         ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import pandas as pd
import numpy as np

# Create a messy dataset
messy_df = pd.DataFrame({
    'name': ['  Alice ', 'Bob', 'CHARLIE', 'diana', 'Eve', 'Bob', 'Alice', None],
    'age': [25, 30, None, 28, 32, 30, 25, 40],
    'salary': ['50000', '60k', '70000', None, '65000', '60000', '50000', '45000'],
    'dept': ['Eng', 'Sales', 'Eng', 'HR', 'Sales', 'Sales', 'Eng', 'Eng'],
    'hire_date': ['2020-01-15', '2019-06-01', '2018-03-20', '2021-09-10',
                  '2020-11-05', '2019-06-01', '2020-01-15', 'invalid'],
    'rating': [4.5, None, 3.8, 4.2, None, None, 4.5, 3.0]
})
print(f"Messy data:\n{messy_df}\n")
print(f"dtypes:\n{messy_df.dtypes}")

In [ ]:
# ============================================================
# 1. Missing Values — Detection
# ============================================================

# Count nulls per column
print(f"Null counts:\n{messy_df.isnull().sum()}\n")

# Percentage of nulls
null_pct = (messy_df.isnull().sum() / len(messy_df) * 100).round(1)
print(f"Null %:\n{null_pct}\n")

# Any nulls at all?
print(f"Has any null: {messy_df.isnull().any().any()}")

# Rows with any null
print(f"\nRows with nulls:\n{messy_df[messy_df.isnull().any(axis=1)]}")

In [ ]:
# ============================================================
# 2. Missing Values — Handling Strategies
# ============================================================

df = messy_df.copy()

# Strategy 1: Drop rows with any null
dropped = df.dropna()
print(f"After dropna: {len(dropped)} rows (was {len(df)})\n")

# Strategy 2: Drop rows where specific columns are null
dropped2 = df.dropna(subset=['name', 'age'])
print(f"Drop if name/age null: {len(dropped2)} rows\n")

# Strategy 3: Fill with constant
df['rating'] = df['rating'].fillna(0)
print(f"Fill rating with 0:\n{df['rating'].tolist()}\n")

# Strategy 4: Fill with mean/median
df['age'] = df['age'].fillna(df['age'].median())
print(f"Fill age with median:\n{df['age'].tolist()}\n")

# Strategy 5: Forward/backward fill (time series)
ts = pd.Series([1, None, None, 4, None, 6])
print(f"Forward fill:  {ts.ffill().tolist()}")
print(f"Backward fill: {ts.bfill().tolist()}")

# Strategy 6: Interpolate (linear)
print(f"Interpolate:   {ts.interpolate().tolist()}")

In [ ]:
# ============================================================
# 3. Duplicates
# ============================================================

df = messy_df.copy()

# Find duplicates (all columns)
print(f"Duplicate rows: {df.duplicated().sum()}\n")

# Find duplicates by subset of columns
print(f"Dup by name+age: {df.duplicated(subset=['name', 'age']).sum()}")

# Show duplicate rows
dupes = df[df.duplicated(subset=['name', 'age'], keep=False)]
print(f"\nDuplicate rows:\n{dupes}\n")

# Drop duplicates
deduped = df.drop_duplicates(subset=['name', 'age'], keep='first')
print(f"After dedup: {len(deduped)} rows (was {len(df)})")

In [ ]:
# ============================================================
# 4. Type Conversion
# ============================================================

df = messy_df.copy()

# String to numeric (handle errors)
df['salary_clean'] = pd.to_numeric(df['salary'], errors='coerce')
print(f"Salary conversion:\n{df[['salary', 'salary_clean']]}\n")
# '60k' becomes NaN (can't convert)

# String to datetime (handle errors)
df['hire_date_clean'] = pd.to_datetime(df['hire_date'], errors='coerce')
print(f"Date conversion:\n{df[['hire_date', 'hire_date_clean']]}\n")
# 'invalid' becomes NaT

# astype for simple conversions
df['age'] = df['age'].fillna(0).astype(int)
print(f"Age as int: {df['age'].tolist()}")

# Category type (saves memory for low-cardinality columns)
df['dept'] = df['dept'].astype('category')
print(f"\nDept dtype: {df['dept'].dtype}")

In [ ]:
# ============================================================
# 5. String Cleaning
# ============================================================

df = messy_df.copy()

# Strip whitespace + standardize case
df['name'] = df['name'].str.strip().str.title()
print(f"Cleaned names: {df['name'].tolist()}\n")

# Replace patterns
# Fix the '60k' salary issue
df['salary'] = df['salary'].str.replace('k', '000', regex=False)
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')
print(f"Fixed salary: {df['salary'].tolist()}\n")

# Extract with regex
emails = pd.Series(['alice@gmail.com', 'bob@yahoo.com', 'charlie@gmail.com'])
domains = emails.str.extract(r'@(\w+)', expand=False)
print(f"Extracted domains: {domains.tolist()}\n")

# Replace using map (recode values)
dept_map = {'Eng': 'Engineering', 'HR': 'Human Resources'}
df['dept_full'] = df['dept'].map(dept_map).fillna(df['dept'])
print(f"Mapped depts: {df['dept_full'].tolist()}")

In [ ]:
# ============================================================
# 6. Outlier Detection & Handling
# ============================================================

np.random.seed(42)
data = pd.DataFrame({
    'value': np.concatenate([np.random.normal(50, 10, 100), [200, -50, 300]])
})

# Method 1: IQR (Interquartile Range)
Q1 = data['value'].quantile(0.25)
Q3 = data['value'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_iqr = data[(data['value'] < lower) | (data['value'] > upper)]
print(f"IQR bounds: [{lower:.1f}, {upper:.1f}]")
print(f"Outliers (IQR): {len(outliers_iqr)}\n")

# Method 2: Z-score
z_scores = (data['value'] - data['value'].mean()) / data['value'].std()
outliers_z = data[z_scores.abs() > 3]
print(f"Outliers (z > 3): {len(outliers_z)}\n")

# Handle: Clip to bounds
data['clipped'] = data['value'].clip(lower, upper)
print(f"Before clip: min={data['value'].min():.0f}, max={data['value'].max():.0f}")
print(f"After clip:  min={data['clipped'].min():.0f}, max={data['clipped'].max():.0f}")

In [ ]:
# ============================================================
# 7. pipe() — Clean, Chainable Pipelines
# ============================================================

# Define cleaning steps as functions
def clean_names(df):
    df = df.copy()
    df['name'] = df['name'].str.strip().str.title()
    return df

def fix_salary(df):
    df = df.copy()
    df['salary'] = df['salary'].str.replace('k', '000', regex=False)
    df['salary'] = pd.to_numeric(df['salary'], errors='coerce')
    return df

def fill_missing(df):
    df = df.copy()
    df['age'] = df['age'].fillna(df['age'].median())
    df['salary'] = df['salary'].fillna(df['salary'].median())
    df['rating'] = df['rating'].fillna(df['rating'].mean())
    return df

def remove_duplicates(df):
    return df.drop_duplicates(subset=['name', 'age'], keep='first')

def drop_invalid(df):
    return df.dropna(subset=['name'])

# Chain them with pipe()
cleaned = (messy_df
    .pipe(clean_names)
    .pipe(fix_salary)
    .pipe(fill_missing)
    .pipe(remove_duplicates)
    .pipe(drop_invalid)
    .reset_index(drop=True)
)

print(f"Original: {len(messy_df)} rows, Cleaned: {len(cleaned)} rows")
print(f"\nCleaned data:\n{cleaned}")
print(f"\nNull check:\n{cleaned.isnull().sum()}")

print("\n💡 pipe() makes cleaning reproducible and readable!")

---

## 🗺️ Data Cleaning Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  DATA CLEANING — Quick Reference                                 ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  MISSING:                                                        ║
║    Detect:  isnull().sum()                                      ║
║    Drop:    dropna()  dropna(subset=['col'])                    ║
║    Fill:    fillna(val)  fillna(df['col'].median())            ║
║    Time:    ffill()  bfill()  interpolate()                    ║
║                                                                   ║
║  DUPLICATES:                                                     ║
║    Find:    duplicated().sum()                                  ║
║    Remove:  drop_duplicates(subset, keep='first')              ║
║                                                                   ║
║  TYPES:                                                           ║
║    to_numeric(errors='coerce')  → NaN for bad values           ║
║    to_datetime(errors='coerce') → NaT for bad dates            ║
║    astype('category')           → save memory                   ║
║                                                                   ║
║  OUTLIERS:                                                       ║
║    IQR:  Q1 - 1.5*IQR, Q3 + 1.5*IQR                          ║
║    Clip: df['col'].clip(lower, upper)                          ║
║                                                                   ║
║  PIPELINE:  df.pipe(f1).pipe(f2).pipe(f3)                      ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | How to handle missing values? | Depends: drop if few, fill with median (numeric) or mode (categorical), ffill for time series |
| 2 | fillna mean vs median? | Median is robust to outliers. Mean for normally distributed data |
| 3 | How to detect outliers? | IQR (Q1-1.5*IQR to Q3+1.5*IQR), z-score (> 3 std), domain knowledge |
| 4 | to_numeric errors='coerce'? | Converts unparseable values to NaN instead of raising error |
| 5 | What is pipe()? | Chain functions: df.pipe(f1).pipe(f2). Each function takes and returns a DataFrame |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • isnull().sum() first — know your data                │
## │  • errors='coerce' for safe type conversion             │
## │  • IQR method for outlier detection                     │
## │  • pipe() for reproducible cleaning pipelines           │
## │  • Always validate: no nulls, correct types, shape      │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Chapter 03 — ML Algorithms: Linear Regression from Scratch**